# Analyse "Agricultural one component transformers" (V4.1)

Import der Module, Basis Funktionen. Nützliche Zusatzbefehle:
- https://status.jupyter.ecobalyse.fr
- http://fplca.ecobalyse.fr 1037639608873
- source EcoBal_P192/bin/activate
- pip install pyvolca --upgrade


## Part 1 TransformerAnalysis : get one component transformers

In [1]:
#Cell "prepare widgets“
from   datetime import datetime
from   IPython.display import display, HTML
import ipywidgets      as     widgets

ui_output         = widgets.Output(layout={'height': '500px', 'overflow':'scroll'})

ui_width = {'width':'560px'}
ui_style = {'description_width':'160px'}
ui_database       = widgets.Text(value="agb4-0_def",      description='Name AGB4-DB:',placeholder='agb40def',disabled=False ,layout= ui_width, style=ui_style   )
ui_flat_database  = widgets.Text(value="flatdb",description='Name flat DB:', placeholder='Type something',disabled=False , layout= ui_width, style=ui_style)
ui_lcia_method    = widgets.Text(value="Ef31",description='Name Method:', placeholder='Type something',disabled=False ,  layout= ui_width , style=ui_style)

ui_folder_input   = widgets.Text(value="pythonIn/",       description='Folder Input:', placeholder='pythonIn/',disabled=False, layout=ui_width, style=ui_style     )

ui_folder_output  = widgets.Text(value="pythonOut/",      description='Folder Output:',placeholder='pythonOut/',disabled=False, layout=ui_width, style=ui_style     )
ui_categories     = widgets.Text(value="194-i01-NewCategories_V5.0_20260910.csv",description='fCategories:',placeholder='Type something',disabled=False, layout= ui_width, style=ui_style    )

ui_result_version = widgets.Text(value="V5.0",            description='Version:',placeholder='Type something',disabled=False, layout=ui_width, style=ui_style     )

ui_file_allconsumers                 = widgets.Text(value="194-o01_AllConsumers_" ,description='File All Transformers:',placeholder='Type something',disabled=False, layout=ui_width, style=ui_style    )
ui_file_one_component_transformers   = widgets.Text(value="194-o02-OneComponentComsumers_" ,description='File one component :',placeholder='Type something',disabled=False, layout=ui_width, style=ui_style    )
ui_file_contribution_results         = widgets.Text(value="194-o03-AnalysisResultsNeu_" ,description="File contribution results", placeholder='Type something',disabled=False , layout=ui_width , style=ui_style   )
ui_file_final_results                = widgets.Text(value="194-o04-FinalAnalysisResult_" ,description='File final results:',placeholder='Type something',disabled=False, layout=ui_width, style=ui_style  )
ui_file_flat_csv                     = widgets.Text(value="194-o05-FlatLCI_" ,description='File flat-csv file:', placeholder='Type something',disabled=False , layout=ui_width, style=ui_style   )
ui_file_comparaison_results          = widgets.Text(value="194-o06-LCIACompareFlat_AGB_" ,description='File comparison results:', placeholder='Type something',disabled=False , layout=ui_width, style=ui_style )
ui_demo                              = widgets.Checkbox(value=False,description='Only Demo (limit to 200 LCI)',disabled=False,indent=False)

ui_button_one = widgets.Button(description="One component")
ui_button_two = widgets.Button(description="Contribution analysis")
ui_button_thr = widgets.Button(description="Impact calculation")

ui_parameter = widgets.VBox([widgets.Label(value="VOLCA parameters:", layout=ui_width ,style={'background':'lightgrey','font_weight':'bold'}),
                             ui_database, ui_flat_database, ui_lcia_method,ui_demo,
                             widgets.Label(value="Input:", layout=ui_width ,style={'background':'lightgrey','font_weight':'bold'}),
                             ui_folder_input,
                             ui_categories,
                             widgets.Label(value="Output: (Version and xlsx/csv will be add to filenames)", layout=ui_width ,style={'background':'lightgrey','font_weight':'bold'}),
                             ui_folder_output, ui_result_version,
                             ui_file_allconsumers, ui_file_one_component_transformers,ui_file_contribution_results, 
                             ui_file_final_results,ui_file_flat_csv, ui_file_comparaison_results,
                             widgets.HBox([ui_button_one,ui_button_two,ui_button_thr]),
                            ], layout={'height': '700px', 'left':'0px', 'width':'570px'})
   
def click_one_component(b):        
    print_to_ui("clear")
    print_to_ui("One Component Transformer")
    try:
        get_rawmats_by_transformers()
    except:
        raise     
    get_one_component_transformers()

def click_contribution(b):
    print_to_ui("clear")
    print_to_ui("Contribution analysis")   
    analyse_contributions()
    print_to_ui("agg")
    try:
        aggreagte_recursion_results()
    except:
        print_to_ui("err agg")
        raise
        
    create_csv_database()  
    
def click_compairison(b):
    print_to_ui("clear")
    print_to_ui("works only, if you manually imported the flat csv file?")   
    calculate_impacts_for_comparison("alles")

ui_button_one.on_click(click_one_component)
ui_button_two.on_click(click_contribution)
ui_button_thr.on_click(click_compairison)



    
ui_main = widgets.HBox([
    ui_parameter, 
    widgets.VBox([
        widgets.Label(value="Output control:", layout={'width':'1000px'} ,style={'background':'lightgrey','font_weight':'bold'}),
        ui_output],layout={'height': '550px', 'width':'1000px'} )
])


def print_to_ui(data, end_char="\n"):
    with ui_output:
        if data == "clear":
            clear_output(wait = False)
        else:
            print( f'{datetime.now()}: {data}', end=end_char )  # bis 19 ist Sekunden end="" or \r

print_to_ui("#Cell 'prepare widgets' compiled.")

In [2]:
# Cell "init, globals" and basic help function

from IPython.display import clear_output  # all three only for formatting output in jupyter lab
from volca import  Client                 #, ActivityInput, TechInput, BioExchange,BioDirection
import openpyxl                           # to write Excelfiles on disk
import csv                                # to handle csv files
import pandas as pd                       # for data transformations in frames

 
def get_volca_db(db_name):              
    """Get connection to local volca, returns db.
       
       Important: Volca desktop has to opened
       for volca-internetserver connection, use volca_db = Client(base_url="https://fplca.ecobalyse.fr", db=databaseName, password="1037639608873") or your private "account".
    """
    try:
        volca_db = Client(base_url="http://127.0.0.1:8080/", db=db_name, password="") 
    
        volca_db.load_database(db_name)
    except:
        print_to_ui("Can't connect to Volca - is Volca-app started?")
        raise Exception("Can't connect to Volca")
        
    return volca_db
    
def read_categories(file_in):
    """Returns the recursion_categories as dataframe.

       Simple transformerAnalysis-Helpfunction. set the index on the link field
    """
    recursion_categories = pd.read_csv(ui_folder_input.value + file_in)
    return recursion_categories

def write_to_excel(file,sheets,dataframes):  # TransformerAnalysis-Helpfunction: to write excel-files on disk
    """TransformerAnalysis-Helpfunction: to write one or more dateframe at once to excel-files on disk.

       Important: provide of sheet and dataframe as list of same length"""
    writer = pd.ExcelWriter(file,engine ='xlsxwriter')
    for i in range(0,len(dataframes)):
        dataframes[i].to_excel(writer, sheet_name = sheets[i])
    writer.close()    

print_to_ui("#Cell 'init, globals' compiled.") # Compilation success message

In [3]:
# Cell "one component transformers"  by top-down (supply_chain): part 1: identify all raw-consumers for each transformer-LCI, V15b 23/08/2026 
def check_chain_level(current_chain, current_id, current_step, level):
    next_step=current_chain[current_chain["from_id"] ==  current_step ]
    if len(next_step) == 0:
        return -11
    else:
                
        next_id =(next_step.iloc[0]["to_id"])
        if next_id == current_id:
            return 0
        elif level  > 10:
            return -10
        else:
            level =level+1
            return check_chain_level(current_chain,current_id, next_id, level)

def get_rawmats_by_transformers():
    """Gets for each food-transformer LCI its agricultural raw material flow by exploiting the supplychain and saves it to excel file.

       Food-transformer LCI corresponds to the inventories of recursion_category "AgricFoodTrans..."
       Agricultural raw material flow are exchanges of recursion_category "AgricFoodRaw..."
       As these raw materials may also be used by other LCI than food-transformers (e.g. wheat for cow-milk production or other raws or agricultural-upstream or even energy) 
       the list of raw materials have to be focussed on direct inputs to food-transformer (sub)steps  
    """
    
    print_to_ui("Starting get_rawmats_by_transformers, executing takes 10 to 15 minutes" )
    
    volca_db             = get_volca_db(ui_database.value)                                                                     # get AGB-database
    recursion_categories = read_categories(ui_categories.value)                                                                # get the lci-list including the recursionClasses
    transformers         = recursion_categories[recursion_categories["recursionClass"].str.startswith( "AgricFoodTrans")]      # get the list of food-transformers to be analysed
    recursion_indexed    = recursion_categories.set_index("process_id") 
    categories_dict      = recursion_indexed["recursionClass"].to_dict()                                                       # make Mapping-dictionary to map recursionclass to inputs
    prodgroups_dict      = recursion_indexed["prodName"].to_dict()                                                             # Mapping-dictionary to map productName (that is the generalised product) to inputs/lci's 
    
    all_results=[]                                                                                                             # init the list-collector for the analysis results as well as two control varialbe for the logging the loop 
    nr=0                                                                     
    max_transformers = len(transformers.index)
    
    if ui_demo.value == True:
        print_to_ui("You're using demo mode!")
        transformers = transformers.head(200)

    for i, transformer in transformers.iterrows():                                                                             # perform analysis for all tranformers
        print_to_ui(f"get_rawmats_by_transformers (main loop): Analysing nr {nr+1} of {max_transformers}, {transformer["process_id"]}, {transformer["prodName"][:50]}...", "\r")
        nr=nr+1
        
        current_inputs = volca_db.get_supply_chain(transformer["process_id"], include_edges=True, max_depth=8)                # get supplyChain over 8 levels to search for agricultural raw mats
        if len(current_inputs.edges) > 0 :                                                                                    # to exclude dummy inventories
            current_edges          = pd.DataFrame(current_inputs.edges)
            current_edges["input"] = current_edges["from_id"].map(categories_dict)                                            # map recursionClasses to inputs/suppliers for identification of agricultural rawMaterials
            current_edges["step"]  = current_edges["to_id"  ].map(categories_dict)                                            # map recursionClasses to transformation-step/consumer as new field step
            current_rawmats        = current_edges[current_edges["input"].str.startswith("AgricFoodRaw")]                     # get only edges having rawMaterials as input/supplier
            current_rawmats        = current_rawmats[ current_rawmats["step"].str.startswith("AgricFoodTrans")]               # get only those rawMaterials that are supplier to transformation steps
            current_rawmat_entries = pd.DataFrame(current_inputs.entries)                                                     # get entries to get amount for this rawMaterials
            current_rawmats.loc[current_rawmats['from_id'].isin(
                current_rawmat_entries['process_id']), 'amount'] = current_rawmats['from_id'].map(
                current_rawmat_entries.set_index('process_id')['scaling_factor'])                                             # add needed amount
            
            transformer_chain      = current_edges.loc[(current_edges["input"].str.startswith("AgricFoodTrans")) 
                & (current_edges["step"].str.startswith("AgricFoodTrans")) ]                                                  # check on continous transformation chain, as an transformation step with a raw material can also be a supplier to a non-food item or energy or...
            
            transformer_chain["step"] = "broken, transformer_chain"                                                           # set as default "broken, next line: is to_id is also in from_id the chain is continous
            transformer_chain.loc[transformer_chain["to_id"].isin(transformer_chain["from_id"]),"step"] = "Agrotransformer_chain"
            transformer_chain.loc[len(transformer_chain)+1] = [transformer["process_id"], 
                                                               "topIn",transformer["process_id"],"topIn",1.0,"topIn","topIn"] # addt the root_product the the valid transformer_chain
            transformer_chain["level"] = - 101
            current_rawmats["level"] = - 101
            z=0
            
            for j, raw_mat in current_rawmats.iterrows():
                current_level= 0
                temp = check_chain_level(transformer_chain, transformer["process_id"], raw_mat["to_id"],current_level)
                current_rawmats.loc[j, "level"] = temp
                z=z+1
            
            current_rawmats            = pd.concat([current_rawmats, transformer_chain], axis=0)                              # join the two result parts
            if len(current_rawmats) > 0 :                                                                                     # add the process_id and generic name to the results       
                current_rawmats["process_id"] = transformer["process_id"]
                current_rawmats["prodName"]   = transformer["prodName"]
                all_results.append(current_rawmats)                                                                           # append it to the list-collector for the analysis results
                
    print_to_ui("get_rawmats_by_transformers: main loop ended, finalising works ....")         
    all_results                  = pd.concat(all_results)                                                                     # join the results of all transformers as dataframe
    all_results["rawmat"]        = all_results["from_id"].map(prodgroups_dict)                                                # add generic product name 
    all_results["location"]      = "isMissing"                                                                                # transformer["location"]
    all_results["intermedphase"] = all_results["to_id"].map(prodgroups_dict)
  
    write_to_excel(ui_folder_output.value +  ui_file_allconsumers.value +  ui_result_version.value + ".xlsx",["all_results"], [all_results])   # save it
    print_to_ui("get_rawmats_by_transformers, done and saved as " + ui_file_allconsumers.value +  ui_result_version.value + ".xlsx"+".")

def get_one_component_transformers():
    """Aggregates the results from former step and saves the final list of the one_component_transfers
        
    Reads the excelfile from former step and aggregates it
    """
    print_to_ui("Start get_one_component_transformers")

    all_results          = pd.read_excel(ui_folder_output.value +  ui_file_allconsumers.value+  ui_result_version.value + ".xlsx")  
    main_transformers    = all_results[all_results["level"] == 0]
    
    recursion_categories = read_categories(ui_categories.value)                        # get recursion_categories
    index_by_group       = recursion_categories.set_index("prodName")                  # set index on name of product/LCI for Mappings
    prodgroups_dict      = index_by_group["prodGroup"].to_dict()                       # mapping-dictionary to map product-group to LCI
    
    main_transformers["productGroup"] = main_transformers["rawmat"].map(prodgroups_dict)           
    unique_prodgroups_of_transformer  = main_transformers.groupby(["prodName", "productGroup"]).agg(
        totAmount =("amount","sum"),
        count     =("prodName","size")
     ).reset_index()                                                                   # aggregate list the products of main_transformers on unique productgroups for a transformer (as e.g. a transfomer may include several types of eggs...)

    
    count_prodgroups_of_transformer = unique_prodgroups_of_transformer.groupby(["prodName"]).agg(
       rawsKg =("totAmount","sum"),
       Count  =("prodName","size")
    ).reset_index()              
    
    unique_prodgroups_of_transformer.loc[unique_prodgroups_of_transformer['prodName'].isin(count_prodgroups_of_transformer['prodName']), 'gesamt'] = unique_prodgroups_of_transformer['prodName'].map(count_prodgroups_of_transformer.set_index('prodName')['rawsKg'])
    unique_prodgroups_of_transformer["part"] = unique_prodgroups_of_transformer["totAmount"]/unique_prodgroups_of_transformer["gesamt"]
    unique_prodgroups_of_transformer["component_part"] = unique_prodgroups_of_transformer["productGroup"] + unique_prodgroups_of_transformer['part'].map( ': {:.2%}'.format)

    count_prodgroups_of_transformer["final_composition"] = unique_prodgroups_of_transformer.groupby(["prodName"])["component_part"].apply('; '.join).reset_index()["component_part"]
    count_prodgroups_of_transformer["rawsKg"] = count_prodgroups_of_transformer['rawsKg'].map( '{:.3} kg raw mat'.format)

    real_one_component_transformers=count_prodgroups_of_transformer.loc[(count_prodgroups_of_transformer["prodName"].str.contains("at packaging") == False) & (count_prodgroups_of_transformer["Count"] == 1) ]
    print([all_results  , main_transformers,  unique_prodgroups_of_transformer,  count_prodgroups_of_transformer,  real_one_component_transformers])
    write_to_excel(ui_folder_output.value +  ui_file_one_component_transformers.value+  ui_result_version.value + ".xlsx",
                   ["all_results","main_transformers","unique_prodgroups","count_prodgroups","real_one_component"], 
                   [all_results  , main_transformers,  unique_prodgroups_of_transformer,  count_prodgroups_of_transformer,  real_one_component_transformers])
   
    print_to_ui("get_one_component_transformers done and saved as " + ui_file_one_component_transformers.value+  ui_result_version.value + ".xlsx")

print_to_ui("#Cell 'one component transformers' compiled." )

## Part 2 TransformerAnalysis : do recursion

In [4]:
#Cell "do contribution analysis"
def do_recursion(lcis, category_dict, result, iteration, volca_db):
    if iteration > 20:
        x=pd.DataFrame([])
        #print(lcis.iloc[0]["process_id"], "i: ERROR !")
        return x #"Fehler"
    
    iteration=iteration+1
    
    for j,lci in lcis.iterrows():
        
        inventory    = volca_db.get_activity(lci["process_id"])
        alloc_factor = 1
        normalize    = inventory.product_amount
        current_products                   = pd.DataFrame(inventory.all_products)
        current_exchanges                  = pd.DataFrame(inventory.exchanges)
        current_exchanges["to_id"]         = lci["process_id"]
        current_exchanges.loc[current_exchanges["role"].isnull(),"role"]    = "biosphereflow"
        current_exchanges                  = current_exchanges[current_exchanges["role"] != "ReferenceProduct"]
        
        current_exchanges["amount"]        = current_exchanges["amount"] / normalize # normalizing the current_exchanges
        current_products["product_amount"] = current_products["product_amount"] / normalize
        
        alloc_factor                       = current_products[current_products["process_id"] == lci["process_id"] ]
        alloc_factor                       = alloc_factor.iloc[0]["allocation_percent"]/100  *lci["amount"]
        current_exchanges["wasAmount"]     = current_exchanges["amount"] * alloc_factor
        current_exchanges["wasAllocF"]     = alloc_factor
        current_exchanges["damount"]       = current_exchanges["amount"] *lci["amount"]
        current_exchanges["inputCat"]      = current_exchanges["flow_name"].map(category_dict)
         
        transfomer_exchanges               = current_exchanges[mask:= (current_exchanges["inputCat"] =="AgricFoodTransformMixes")]
        remaining_exchanges                = current_exchanges[~mask]
        result                             = remaining_exchanges
        if len(transfomer_exchanges)  > 0:
            for t, current_transformer in transfomer_exchanges.iterrows():
                next_step          = pd.DataFrame({'process_id': [current_transformer.target_process_id],'amount': [current_transformer.damount]})
                next_contributions = do_recursion(next_step, category_dict, result, iteration, volca_db)
                result = pd.concat([next_contributions, result])
    return result
    
def perform_analysis(current_product, volca_db):
    
    current_contributions = pd.DataFrame([])    
    recursion_categories  = read_categories(ui_categories.value)       
    
    recursion_categories = recursion_categories.set_index("prodName")
    category_dict        = recursion_categories["recursionClass"].to_dict()
    
    root_product         = pd.DataFrame([],columns=["process_id","amount"])
    root_product.loc[0]  = [current_product,1.0]       # the real referenceflow is catch in the function
    iteration = 1   
    current_contributions = do_recursion(root_product, category_dict, current_contributions, iteration, volca_db)
    
    current_contributions = current_contributions.rename(columns={'target_process_id':'from_id', 'flow_id':'id_exchange'})
    current_product_infos = volca_db.get_activity(current_product)
    current_coproducts    = pd.DataFrame(current_product_infos.all_products)
    current_coproducts["product_amount"] = current_coproducts["product_amount"] /current_product_infos.product_amount
 
    main_amount=1.0
    
    if len(current_coproducts) > 1: 
        current_coproducts          = current_coproducts.drop(columns=["activity_name","allocation_formula"])
        current_coproducts["to_id"] = current_product
        current_coproducts["role"]  = "coproducts"
        main_amount                 = current_coproducts.loc[current_coproducts["process_id"] == current_product]["product_amount"].values[0]
        current_coproducts          = current_coproducts.rename(columns={'process_id':'from_id','product_amount':'amount','product_unit':'unit', 'allocation_percent':'alloc_factor'})
        current_contributions       = pd.concat([current_contributions,current_coproducts])
    
    current_contributions["forProduct"] = current_product
    current_contributions["mainAmount"] = str(main_amount)
    return current_contributions
    
def read_transformer_list():
    one_component_transformers        = pd.read_excel(ui_folder_output.value + ui_file_one_component_transformers.value+  ui_result_version.value + ".xlsx",sheet_name="real_one_component")     
    return one_component_transformers
    
def analyse_contributions(test="nein"):
    
    print_to_ui("Start analyse_contributions")
    volca_db             = get_volca_db(ui_database.value)                                                                     # get AGB-database
    
    all_process_ids             = volca_db.search_activities(limit=30000)
    process_dict                = {lci.product_name: lci.process_id for lci in all_process_ids}
    recursion_categories        = read_categories(ui_categories.value)                                                          
    flows_of_interest           = recursion_categories[recursion_categories["recursionClass"].isin(["BiowasteOrByproductWaste","EnergyElectricity","EnergyHeat","WaterRaw"])]
    contributors_of_interest    = recursion_categories[recursion_categories["recursionClass"].isin(["AgricFoodTransformMixes"])]
    
    one_component_transformers  = read_transformer_list()
    one_component_transformers["process_id"] = one_component_transformers["prodName"].map(process_dict)
    
    all_contributions=[]
        
    for i, transformer in one_component_transformers.iterrows():   

        print_to_ui(f"newContributionAnalysis: Analysiere Nr {i} = {transformer.prodName[:60]}", "\r")
        current_contributions = perform_analysis(transformer["process_id"], volca_db)  
        all_contributions.append(current_contributions)
        
    all_contributions=pd.concat(all_contributions)

    category_types_file = ui_folder_input.value +  "194-i02-CategoryMapping_V5.0_20260812.csv"
    category_types      = pd.read_csv(category_types_file)
    category_types      = category_types.set_index("Category")
    category_dict       = category_types["AggCat"].to_dict()
    process_dict        = {lci.process_id: lci.product_name for lci in all_process_ids}
    unit_dict           = recursion_categories["unit"].to_dict()    
    
    all_contributions["transformationStep"]        = all_contributions["to_id"].map(process_dict)
    all_contributions["genericTransformationStep"] = all_contributions["transformationStep"].str.split(" {", n=1, expand=True )[0]
    all_contributions["processToAnalyse"]          = all_contributions["forProduct"].map(process_dict)
    all_contributions["input"]                     = all_contributions["from_id"].map(process_dict)
    all_contributions["inputType"]                 = all_contributions["inputCat"].map(category_dict)
    all_contributions.loc[all_contributions["inputType"].isnull(),"inputType"] = all_contributions.loc[all_contributions["inputType"].isnull(),"role"]
    all_contributions["neededAmount"]              = all_contributions["amount"]
    all_contributions["mainUnit"]                  = all_contributions["forProduct"].map(unit_dict)
    
    write_to_excel(ui_folder_output.value +  ui_file_contribution_results.value+  ui_result_version.value + ".xlsx",["all_contributions","recursion_categories","category_types"],[all_contributions,recursion_categories,category_types])
    
    print_to_ui(f"analyse_contributions done and saved as {ui_file_contribution_results.value+  ui_result_version.value + ".xlsx"} End interessiern tut nur depth 1! --------- ")
    
print_to_ui("#Cell 'do contribution analysis' compiled.")

In [5]:
#Cell "aggregate results"

def aggreagte_recursion_results():
    print_to_ui("Start aggreagte_recursion_results")
    
    recursion_results            = pd.read_excel(ui_folder_output.value + ui_file_contribution_results.value+  ui_result_version.value + ".xlsx",sheet_name="all_contributions")     
    recursion_results["detFlow"] = "_"
    recursion_results            = recursion_results.sort_values(["forProduct","to_id","inputType"])
    
    result_row          = {"processToAnalyse":[],"genericTransformationStep":[],"inputType":[],"inputLCI":[], "aggAmount":[],"Depth":[],"detailedFlows":[],"mainUnit":[],"mainAmount":[] }
    aggregated_results  = pd.DataFrame(result_row)
    
    count_inputs   = 0
    collect_inputs = ""
    add_amount     = 0
  
    generic_transformation_step = recursion_results.iloc[0]["genericTransformationStep"]
    input_type                  = recursion_results.iloc[0]["inputType"]
    result_row["Depth"]         = 1 #recursion_results.iloc[0]["Depth"]
    
    for i,current_row in recursion_results.iterrows():
        
        if current_row["genericTransformationStep"] == generic_transformation_step:
            if current_row["inputType"] == input_type:
                count_inputs = count_inputs + 1 
            else:
                result_row["detailedFlows"]= str(count_inputs) + " inputs: " + collect_inputs
                result_row["aggAmount"]= add_amount
                row_number = len(aggregated_results)
                aggregated_results.loc[row_number]=result_row
                count_inputs = 0
                collect_inputs=""
                add_amount=0
                input_type=current_row["inputType"]
        else:
            
            result_row["detailedFlows"]=  str(count_inputs - 1) + " inputs: " + collect_inputs
            result_row["aggAmount"]= add_amount
            row_number=len(aggregated_results)
      
            aggregated_results.loc[row_number]=result_row
            result_row["Depth"]= 1 #current_row["Depth"]
            collect_inputs=""
            add_amount=0
            count_inputs = 0
            generic_transformation_step = current_row["genericTransformationStep"]
            input_type=current_row["inputType"]
      
        result_row["processToAnalyse"]= current_row["processToAnalyse"]
        result_row["mainAmount"]= current_row["mainAmount"]
        result_row["mainUnit"]= current_row["mainUnit"]
        result_row["inputType"]= current_row["inputType"]
        result_row["inputLCI"]= current_row["input"]
        result_row["genericTransformationStep"]= current_row["genericTransformationStep"]
        if current_row["role"] == "biosphereflow" :
            sName="flow_name"
            sAmount = "damount"
        elif current_row["role"] == "coproducts" :
            sName = "input"
            sAmount = "neededAmount"
        else:
            sName = "input"
            sAmount = "damount"
        add_amount=add_amount + current_row[sAmount]
        collect_inputs = collect_inputs  +str(current_row[sName]) + ": " + str(current_row[sAmount])+ " " +str(current_row["unit"]) +"\n" # :: \n

    writer = pd.ExcelWriter(ui_folder_output.value + ui_file_final_results.value+  ui_result_version.value + ".xlsx",                            engine ='xlsxwriter')

    aggregated_results.to_excel(writer, sheet_name ='aggregated')
    writer.close()    
    print_to_ui("fertig aggreagte_recursion_results")

print_to_ui("#Cell 'aggregate results' compiled.")


## Part 3 TransformerAnalysis : Make CSV and results

In [6]:
#Cell "Make Csv" 

def make_one_csv_inventory(current_exchanges,csv_file):
    exchange_type = 0
    current_category="[]"
    csv_metadata_lines = ["\n","Status\n","\n","\n","Time period\n","\n","\n","Geography\n","\n","\n", "Technology\n","\n","\n",
                          "Representativeness\n","\n","\n","Infrastructure\n","No\n","\n","Record\n","Ecolysis - Peter Koch\n","\n",
                          "Generator\n","Ecolysis - Peter Koch\n","\n","External documents\n","\n","\n","Literature references\n","\n","\n",
                          "Collection method\n","\n","\n", "Data treatment\n","\n","\n","Verification\n","\n","\n","Comment\n","\n","\n",
                          "Allocation rules\n","\n","\n","System description\n","\n"]
 
    #colList = csv_inventory_start
    
    for i, current_exchange in current_exchanges.iterrows():
        if exchange_type == 0:   ### ist recursion_results in mainxxx  
            csv_file.write( "pk_" + current_exchange["processToAnalyse"] +"\n" )
            
            for i in csv_metadata_lines:
                csv_file.write(i)
            
            csv_file.write("\nProducts\n")
            csv_file.write("\"pk_" + str(current_exchange["processToAnalyse"]) + "\";kg;1;100;Compost;Agricultural.Ecobalyse;created by python V1.0\n")
            exchange_type = 1
            
        if current_exchange["role"] == "biosphereflow":
            sub_category = ""
            exchange_type = 2
            compartment_infos = str(current_exchange["compartment"]).split("'")
            
            if current_category != compartment_infos[3]:
                if compartment_infos[3] == "resource":
                    csv_file.write("\n\nResources\n")
                else:
                    csv_file.write("\n\nEmissions to " + compartment_infos[3]+"\n")
                current_category = compartment_infos[3]
            
            if compartment_infos[6] == ": None}":
                sub_category="unspecified"
            else:
                sub_category=compartment_infos[7]

            csv_file.write("\"" + str(current_exchange["flow_name"])+"\";" + str(sub_category) +";"+ current_exchange["unit"]+";"+str(current_exchange["damount"]) +";Lognormal;1.573;0;0;(3,3,5,3,1,na)\n")
        
        elif current_exchange["role"] == "coproducts": #vor inputCat!
            exchange_type=3

        elif current_exchange["role"] == "FinalWasteFlow":
            if exchange_type < 4:
                csv_file.write("\nFinal waste flows\n")
                exchange_type=4
            csv_file.write("\"" + str(current_exchange["flow_name"]) + "\";;" + current_exchange["unit"] +";"+ str( current_exchange["damount"]) +";Lognormal;1.573;0;0;(3,3,5,3,1,na)\n")
           
        elif current_exchange["role"] == "Input":
            if exchange_type < 5:
                csv_file.write("\nMaterials/fuels\n")
                exchange_type=5
            csv_file.write("\"" + str(current_exchange["input"]) + "\";" + current_exchange["unit"] +";"+ str( current_exchange["damount"]) +";Lognormal;1.573;0;0;(3,3,5,3,1,na)\n")
                    
    csv_file.write( "\n\nEnd\n\n" )
  
    return

def create_csv_database():
    csv_file_start      = ["{SimaPro 9.0.0.49}","{processes}","{Date: 21.08.2026}","{Time: 14:25:39}","{CSV Format version: 9.0.0}","{CSV separator: Semicolon}","{Decimal separator: .}","{Date separator: .}",
                           "{Short date format: dd.MM.yyyy}\n", "{Export platform IDs: No}","{Skip empty fields: No}","{Convert expressions to constants: Yes}","{Selection: Selection (1)}",
                           "{Include sub product stages and processes: No}\n","\n"]
    
    csv_inventory_start = ["Process\n","\n","Category type\n","material\n","\n","Process identifier\n","\n","\n","Type\n","Unit process\n","\n","Process name\n"]
 
            
    print_to_ui("Start, create_csv_database")
    all_inventories = pd.read_excel(ui_folder_output.value + ui_file_contribution_results.value+  ui_result_version.value + ".xlsx" )
    all_inventories = all_inventories.sort_values(by=["direction","role","forProduct"])

    all_inventories = all_inventories.groupby("forProduct")
    
    csv_collector = csv_file_start
    print_to_ui("start looping")
    
    csv_database_file = ui_folder_output.value +  ui_file_flat_csv.value +  ui_result_version.value + ".csv"
    with open(csv_database_file, "w", encoding="utf-8") as csv_file:
    
        for i in csv_file_start:
            csv_file.write(i + "\n")  
    
        for j, current_lci in all_inventories:
            print_to_ui("handling: " + j, "\r")    
            
            for i in csv_inventory_start:
                csv_file.write(i)  #Start of CSV
                
            make_one_csv_inventory(current_lci, csv_file)
                

    print_to_ui("ENDE, mach CSV--------- ")  

print_to_ui("#Cell 'Make Csv' compiled.")

In [7]:
#Cell "calculate impacts for comparison"
def makeDataFrame(res):
    colnames=["product"]
    for impact in res:
        colnames.append(impact.method_name)
    emptyDF = pd.DataFrame(columns=colnames)
    return emptyDF

def calculate_impacts_for_comparison(calculation_set = "all"):
    """Impacts.

       calucation_set = AGB or flat or all = both
    """
    
      
    
    print_to_ui("calculate_impacts_for_comparison - kann eine halbe Stunde dauern: " + str(ui_flat_database.value))
    
    flat_db = Client(base_url="http://127.0.0.1:8080/", db=str(ui_flat_database.value),password="") #, password="1037639608873")
    flat_db.load_database(str(ui_flat_database.value))
 
    all_inventories = flat_db.search_activities(page_size=30000)    #get all LCIs from out and agb4
    lci_list=[]
    
    create_dataframe = "yes"

    print_to_ui("Starting with flat db ..........")
    for lci in all_inventories:
        if lci.product_name.startswith("pk") == True :
            print_to_ui(lci.product_name[:60], "\r")
            if calculation_set != "AGB":    
                impacts = flat_db.get_impacts_batch(lci.process_id, collection=ui_lcia_method.value)
                if create_dataframe == "yes": 
                    all_results = makeDataFrame(impacts.results)
                    create_dataframe =" does already exist"
                all_results.loc[len(all_results)] = [lci.product_name] + [scores.score for scores in impacts.results]
            lci_list.append(lci.product_name[3:1000])
        
    print_to_ui(f"End of recursion_results (flat db): {len(lci_list)}")
    
    if calculation_set != "flat":
        volca_db =get_volca_db(ui_database.value)
        print_to_ui("Starting with AGB4 db ..........")
        for lci in lci_list:
            process_id = volca_db.search_activities(product=lci, exact=True)    #get all LCIs from out and agb4
            print_to_ui(lci[:60] + str(process_id[0].process_id),"\r")
            impacts = volca_db.get_impacts_batch(process_id[0].process_id, collection=ui_lcia_method.value)
            if create_dataframe == "yes": 
                all_results = makeDataFrame(all_results.results)
                create_dataframe =" does already exist"

            all_results.loc[len(all_results)] = [lci] + [scores.score for scores in impacts.results]
        
        print_to_ui(f"End of recursion_results: {len(lci_list)}")
    
    write_to_excel(ui_folder_output.value + ui_file_comparaison_results.value+  ui_result_version.value + ".xlsx",["all_results"],[all_results])

print_to_ui("#Cell 'calculate impacts for comparison' compiled.")

# Main

In [8]:

display( ui_main )
print_to_ui("All compiled, execution possible.")

# das für tests

In [9]:
# installations test swith to RAW, if tested succesfully
def test_requirements():
    import sys
    version = f"Phyton {sys.version}"
    installer = !conda --version
    if "command not found: conda" in installer[0]:
        installer = !pip --version
    del sys
    print(f"Installed python and package-installer:\n{version}\n{installer[0]}")
    installer=installer[0][:3]
    print("============================\nChecking modules")
    
    try:
        import volca as test 
    except:
        if input("Volca is not installed - enter y to install") == "y":
            if installer=="pip":
                %pip install pyvolca
            else:
                %conda install pyvolca

    try:
        import ipywidgets as test
        print(test.__name__, test.__version__)
    except:
        if input("ipywidgets is not installed - enter y to install") == "y":
            if installer=="pip":
                %pip install ipywidgets
            else:
                %conda install ipywidgets

    try:
        import pandas as test
        print(test.__name__, test.__version__)
    except:
        if input("pandas is not installed - enter y to install") == "y":
            if installer=="pip":
                %pip install pandas
            else:
                %conda install pandas

    try:
        import openpyxl as test 
        print(test.__name__, test.__version__)
    except:
        if input("openpyxl is not installed - enter y to install") == "y":
            if installer=="pip":
                %pip install openpyxl
            else:
                %conda install openpyxl
    print("End Checking\n============================")

    print("""
Shall output: 

Installed python and package-installer:
Phyton 3.13.14 (v3.13.14:fd17997c386, Jun 10 2026, 08:55:00) [Clang 21.0.0 (clang-2100.1.1.101)]
pip 26.2.1 from /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip (python 3.13)
============================
Checking modules
ipywidgets 8.1.9
pandas 3.0.5
openpyxl 3.1.5
End Checking
============================""")


In [ ]:
#manual launching of functions. if wanted: uncomment and run. dont forget to recomment after use
print("uncomment and recommeant and set condition to tru")
    
if 1 == 1 :
    print(" ")
    #get_rawmats_by_transformers()    
    #get_one_component_transformers()
    #analyse_contributions()    
    #aggreagte_recursion_results()
    #create_csv_database()    
    #calculate_impacts_for_comparison("alles")
